# Local Zero-Shot Text Classification (Hugging Face)

Notebook ini menggunakan model `facebook/bart-large-mnli` untuk melakukan klasifikasi teks tanpa menggunakan API eksternal (OpenAI/Gemini).

In [ ]:
# Langkah 1: Fix Error DLL (WinError 1114)
# Kita akan paksa install Torch versi CPU agar tidak terjadi bentrok dengan driver GPU/DLL Windows.
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cpu
!pip install pandas transformers==4.38.2 scikit-learn tqdm

In [ ]:
import sys
import os

print("--- DIAGNOSA SISTEM ---")
try:
    import torch
    import pandas as pd
    from transformers import pipeline
    from tqdm import tqdm
    print("✅ Semua library berhasil di-import.")
except Exception as e:
    print(f"❌ ERROR: Masih terjadi masalah: {e}")
    print("Silakan RESTART KERNEL notebook ini setelah instalasi di atas selesai.")
    sys.exit()

print(f"Python: {sys.version.split()[0]}")
print(f"Torch version: {torch.__version__}")

model_name = "facebook/bart-large-mnli"

# Inisialisasi Pipeline (Dipaksa pakai CPU agar stabil)
print(f"\nMemuat model '{model_name}'... ")
print("⚠️ PERHATIAN: Download 1.6GB (Sabar ya!).")

try:
    classifier = pipeline(
        "zero-shot-classification", 
        model=model_name,
        device=-1 # Gunakan CPU
)
    print(f"✅ Model BERHASIL dimuat di CPU!")
except Exception as e:
    print(f"❌ ERROR Model Gagal dimuat: {e}")
    print("\nMencoba menggunakan MODEL CADANGAN yang lebih ringan (DistilBART)... ")
    classifier = pipeline(
        "zero-shot-classification", 
        model="valhalla/distilbart-mnli-12-3",
        device=-1
    )
    print("✅ Model CADANGAN berhasil dimuat.")

def zero_shot_classify(text, labels):
    if pd.isna(text) or str(text).strip() == "":
        return "BUKAN"
    try:
        result = classifier(text, candidate_labels=labels)
        return result['labels'][0]
    except Exception as e:
        return f"ERROR: {str(e)}"

## Memuat Data dan Melakukan Klasifikasi

In [ ]:
input_file = r"C:\Users\mahesa\Downloads\evaluasi_30.csv"
candidate_labels = ["PROMOSI", "AJAKAN", "BUKAN"]
output_file = "zero_shot_results.csv"

if not os.path.exists(input_file):
    print(f"⚠️ File tidak ditemukan di: {input_file}")
else:
    df = pd.read_csv(input_file)
    if 'tweet_text' not in df.columns:
        print(f"❌ Kolom 'tweet_text' tidak ada! Kolom tersedia: {df.columns.tolist()}")
    else:
        print(f"Memulai klasifikasi untuk {len(df)} data...")
        results = []
        for text in tqdm(df['tweet_text']):
            label = zero_shot_classify(text, candidate_labels)
            results.append(label)

        df['predicted_label'] = results
        df.to_csv(output_file, index=False)
        print(f"\n✅ Selesai! Hasil disimpan ke: {output_file}")
        print(df[['tweet_text', 'predicted_label']].head())

## 📊 Laporan Akurasi

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

target_col = 'label' if 'label' in df.columns else 'label_llm' if 'label_llm' in df.columns else None

if 'predicted_label' in df.columns and target_col:
    mask = ~df['predicted_label'].str.contains("ERROR", na=False)
    eval_df = df[mask].copy()
    y_true = eval_df[target_col].astype(str).str.strip()
    y_pred = eval_df['predicted_label'].astype(str).str.strip()
    print(f"Target Column: {target_col}")
    print(classification_report(y_true, y_pred))
    acc = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy: {acc*100:.2f}%")
else:
    print("❌ Kolom target tidak ditemukan.")